In [6]:
import os 
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore


In [18]:
import sys
from pathlib import Path

# Backend folder ko Python path me add karo
BACKEND_DIR = Path.cwd().parent

sys.path.insert(0, str(BACKEND_DIR))

print(BACKEND_DIR)

e:\langchain\backend\app


In [19]:
load_dotenv()

True

In [20]:
loader = PyPDFLoader('../rag/ashish.pdf')
documents = loader.load()

In [21]:
print('lengths of documens' , len(documents))

lengths of documens 1


In [22]:
test_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 70 , 
    chunk_overlap = 10 

)

In [23]:
chunks = test_splitter.split_documents(documents=documents)

In [24]:
for chunk in chunks:
    print(chunk.page_content)

Ashish Maurya Location: Maharashtra, Mumbai, India
Ashishmaurya.com | LinkedIn | GitHub | Leetcode Email:
Email: ramashishmaurya738@gmail.com | Mobile: 9769295699
SUMMARY
I am an aspiring AI/ML Engineer with strong skills in Python, Machine
Machine Learning, Deep Learning, and NLP. I have hands-on
experience building end-to-end ML pipelines, training and evaluating
models, and deploying AI-powered applications. I am
proficient in Scikit-learn, TensorFlow, Hugging Face Transformers,
and SQL, and passionate about solving real-world
problems through intelligent systems, predictive modeling, and
and data-driven decision-making.
SKILLS
Programming: Python, R, SQL
Machine Learning & AI: Deep Learning, NLP , LLMs, Model Optimization
Frameworks & Libraries: FastAPI, PyTorch, Scikit-learn, LangChain
Tracking & Document Processing: MLFLOW , PyMuPDF, PyPDF
Databases: SQL , MongoDB , PostgreSQL
Tools: Git, GitHub, Jupyter Notebook, Docker
EDUCATION
EDUCATION
University of Mumbai Maharashtra, Mumba

In [25]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

ModuleNotFoundError: No module named 'app'

In [30]:
from qdrant_client import QdrantClient

In [31]:
clients = QdrantClient(
    url=os.getenv("QDRANT_URL") , 
    api_key=os.getenv("QDRANT_API_KEY")
)

print('sucessfully connected to the qdrant cloud')

sucessfully connected to the qdrant cloud


In [36]:
vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    client=clients,
    collection_name="rag_assistant"
)

TypeError: Client.__init__() got an unexpected keyword argument 'client'

In [35]:
print(clients.get_collections())

collections=[]


In [38]:
vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    collection_name="rag_assistant",
)

In [39]:
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print(client.get_collections())

collections=[CollectionDescription(name='rag_assistant')]


In [59]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3} 
)

In [66]:
print(retriever)

tags=['QdrantVectorStore', 'GoogleGenerativeAIEmbeddings'] vectorstore=<langchain_qdrant.qdrant.QdrantVectorStore object at 0x000001BB07A5C9D0> search_kwargs={'k': 3}


In [72]:
docs = retriever.invoke("which university am i from ")

In [73]:
for i, doc in enumerate(docs):
    print(f"Chunk {i+1}")
    print(doc.page_content)
    print("=" * 80)

Chunk 1
EDUCATION
University of Mumbai Maharashtra, Mumbai, India
Chunk 2
EDUCATION
University of Mumbai Maharashtra, Mumbai, India
Chunk 3
Bachelor of Data Science Aug 2022 – Jun 2025
PROJECTS


In [74]:
context = "\n\n".join([doc.page_content for doc in docs])

question = "name of the my 2 projects"

prompt = f"""
You are an AI assistant.

Use only the given context.

Context:
{context}

Question:
{question}

Answer:
"""

In [63]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [77]:
llm = ChatGoogleGenerativeAI(
     model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)

In [78]:
response  = llm.invoke(prompt)
print(response.content)

I cannot tell you the names of your 2 projects as they are not mentioned in the provided context. The context only has a "PROJECTS" heading without any specific project details listed under it.
